# Save RR curves as netCDF files

The GBD RR curves can be downloaded [here](https://doi.org/10.6069/vkdr-qy60) in csv format. This script converts them to netCDF.

In [ ]:
import os
import pandas as pd

In [ ]:
# === Setup ===
DIR = "/glade/work/awells/air_quality/GBD21/RR_curves/"

# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

# === Loop through variables ===
for var in health_vars:
    file = f"IHME_GBD_2021_AIR_POLLUTION_1990_2021_PM_RR_{var}_MEAN_Y2022M01D31.CSV"
    file_path = os.path.join(DIR, file)
    df = pd.read_csv(file_path)

    # set a multi-index exposure
    df_indexed = df.set_index(["exposure"])
    # convert to xarray Dataset
    ds = df_indexed.to_xarray()

    cause_ID = ds.cause[0].values
    ds = ds.drop_vars("cause")
    description = (
        "GBD 2021 Air Pollution Risk Curves for PM2.5. - "
        "Institute for Health Metrics and Evaluation (IHME). "
        "Global Burden of Disease Study 2021 (GBD 2021) Air "
        "Pollution Exposure Estimates and Risk Curves 1990-2021. "
        "Seattle, United States of America: Institute for Health "
        "Metrics and Evaluation (IHME), 2024."
    )
    ds.attrs["cause_ID"] = str(cause_ID)
    ds.attrs["cause_variable"] = var
    ds.attrs["description"] = description

    out_file = f"IHME_GBD_2021_AIR_POLLUTION_1990_2021_PM_RR_{var}.nc"
    out_path = os.path.join(DIR, out_file)
    ds.to_netcdf(out_path)
    print(f"Saved {out_path}")